# Biblioteka Seaborn

Seaborn to biblioteka wykorzystywana do statystycznych wizualizacji danych. Została zbudowana na bazie biblioteki matplotlib i jednocześnie została zintegrowana do struktur danych udostępnianych przez bibliotekę pandas. 

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

In [ ]:
athlete_events = pd.read_csv('athlete_events.csv', sep=';')

athlete_events.info()

--------------------
### Wykresy relacyjne

Wykresy relacyjne przedstawiają relacje pomiędzy danymi. Funkcja `relplot` znajduje się na poziomie *figure* (poziom wyższy) i korzysta z funkcji z poziomu *axes* (poziom niższy): `scatterplot` oraz `lineplot`. Do wizualizacji danych można wykorzystywać wszystkie powyższe funkcje z odpowiednimi parametrami.

##### ⭐ Zadanie 1: 

Przygotuj wykres punktowy (`scatterplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Zapoznaj się z funkcjami `set_theme` i `set_style` oraz ich parametrami. Wykorzystaj je, żeby dostosować wygląd swojego wykresu. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
usa_medals = athlete_events.loc[
    (athlete_events['NOC'] == 'USA') &
    (athlete_events['Year'] >= 1950) &
    (athlete_events['Year'] <= 2017) &
    (athlete_events['Medal'].notna())
]

top_sport = usa_medals.groupby('Sport')['Medal'].size().idxmax()
top_sport_data = usa_medals[usa_medals['Sport'] == top_sport]
medals_per_year = top_sport_data.groupby(['Year', 'Sex']).size().reset_index(name='Medal Count')

sns.set_theme(
    style="darkgrid",
    palette="deep",
    font="sans-serif",
    font_scale=1.1
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=medals_per_year,
    x='Year',
    y='Medal Count',
    hue='Sex',
    color='#1f77b4',
    s=100
)

ax.set_yticks(range(0, 51, 5))
ax.set_xticks(range(1952, 2017, 4))
plt.title(f'Liczba medali USA w dyscyplinie: {top_sport} (1952–2016)', fontsize=14, pad=20)
plt.xlabel('Rok', labelpad=10, fontsize=12)
plt.ylabel('Liczba medali', labelpad=10, fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### ⭐ Zadanie 2:

Przygotuj wykres liniowy (`lineplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Zapoznaj się z parametrem `hue` oraz dobierz dla niego właściwą wartość mając na uwadze wizualizację określonej liczby serii danych. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i legenda). 

In [ ]:
medals = athlete_events[
    athlete_events['Medal'].notna() &
    (athlete_events['NOC'].isin(['USA', 'CHN', 'RUS'])) &
    (athlete_events['Year'] >= 1994) &
    (athlete_events['Year'] <= 2017)
]

medal_counts = (
    medals.groupby(['Year', 'NOC'])
    .size()
    .reset_index(name='Medal Count')
)

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(
    data=medal_counts,
    x='Year',
    y='Medal Count',
    hue='NOC',
    marker='o'
)
ax.set_xticks(range(1994, 2017, 2))

plt.title('Liczba medali zdobytych przez USA, Chiny i Rosję (1994–2016)', fontsize=14, pad=20)
plt.xlabel('Rok', labelpad=10, fontsize=12)
plt.ylabel('Liczba medali', labelpad=10, fontsize=12)

legend_elements = [
    Line2D([0], [0], marker='o', color='blue', label='Chiny', markersize=8, linewidth=2),
    Line2D([0], [0], marker='o', color='orange', label='Rosja', markersize=8, linewidth=2),
    Line2D([0], [0], marker='o', color='green', label='USA', markersize=8, linewidth=2)
]

plt.legend(
    title='Kraj',
    handles=legend_elements
)
plt.tight_layout()
plt.show()

##### ⭐ Zadanie 3:

Przygotuj wykres liniowy (`relplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych, gdzie każda seria danych będzie przedstawiona na osobnym podwykresie (`subplot`) jednego obrazu. Zapoznaj się z parametrami `row` i `col` oraz dobierz dla nich właściwą wartość mając na uwadze wizualizację określonej liczby serii danych na osobnych podwykresach. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i ewentualnie legenda). 

In [ ]:
medals = athlete_events[
    athlete_events['Medal'].notna() &
    athlete_events['Weight'].notna() &
    athlete_events['Sex'].notna()
]

filtered = medals.loc[medals['Sport'] == 'Taekwondo']
filtered['Event'] = filtered['Event'].str.replace("Men's ", "")
filtered['Event'] = filtered['Event'].str.replace("Women's ", "")

grouped = filtered.groupby(['Year', 'Event', 'Sex'])['Weight'].mean().reset_index()
grouped.sort_values('Weight', ascending=True, inplace=True)

sns.set_theme(style="whitegrid")

color_map = {"M": "#3498db", "F": "#e74c3c"}
legend_elements = [
    Line2D([0], [0], marker='o', markerfacecolor=color_map["M"],
           markersize=5, color=color_map["M"], label='Mężczyźni'),
    Line2D([0], [0], marker='o', markerfacecolor=color_map["F"],
           markersize=5, color=color_map["F"], label='Kobiety')
]

g = sns.relplot(
    data=grouped,
    x='Year',
    y='Weight',
    hue='Sex',
    col='Event',
    row='Sex',
    kind='line',
    marker='o',
    palette=color_map,
    facet_kws={'sharey': False, 'sharex': True},
    height=4,
    aspect=1.2,
    legend=False
)

g.set_titles(col_template="{col_name}")
g.set_axis_labels("Rok", "Średnia waga (kg)")

for ax in g.axes.flat:
    ax.set_xticks(range(2000, 2017, 4))

plt.legend(title='Płeć', handles=legend_elements, loc='best')
plt.tight_layout()
plt.show()
# wybrac ktore maja najwiekszy potencjał/pobawic sie kolorami moze/ jakies fajerwerki niech beda

##### ⭐ Zadanie 4:

Przygotuj wykres bąbelkowy (`relplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Wykorzystaj rozmiary i kolory markerów do zaprezentowania większej liczby informacji. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i opis skali koloru). 

In [ ]:
subset = athlete_events[
    (athlete_events['Sex'] == 'M') &
    (athlete_events['Season'] == 'Winter') &
    (athlete_events['Height'].notna()) &
    (athlete_events['Weight'].notna())
]

top_sports = (
    subset.groupby('Sport')
    .agg({
        'Height': 'mean',
        'Weight': 'mean',
        'ID': 'count',
        'Medal': lambda x: x.notna().mean() * 100
    })
    .rename(columns={'ID': 'Count', 'Medal': '% Medalists'})
    .sort_values(by='Count', ascending=False)
    .head(20)
    .reset_index()
)

bubble = sns.relplot(
    data=top_sports,
    x='Weight',
    y='Height',
    size='Count',
    hue='% Medalists',
    palette='viridis',
    sizes=(50, 1000),
    alpha=0.7,
    marker='o',
    edgecolor='black',
    height=7,
    aspect=1.5,
    legend=False
)

for i in range(top_sports.shape[0]):
    count = top_sports['Count'][i]
    offset = 0.15 if count < 2000 else 0.25

    plt.text(
        top_sports['Weight'][i],
        top_sports['Height'][i] + offset,
        top_sports['Sport'][i],
        fontsize=9,
        ha='center',
        va='bottom'
    )

sc = plt.scatter(
    top_sports['Weight'],
    top_sports['Height'],
    c=top_sports['% Medalists'],
    cmap='viridis',
    s=0
)
cbar = plt.colorbar(sc)
cbar.set_label('% Medalistów')

bubble.set_axis_labels("Średnia waga (kg)", "Średni wzrost (cm)")
plt.title("Profil fizyczny sportowców w 15 najpopularniejszych dyscyplinach\n(Mężczyźni, Zimowe Igrzyska)", fontsize=14)

plt.tight_layout()
plt.show()

--------------------
### Wykresy dystrybucji

Wykresy dystrybucji przedstawiają w jawny sposób rozkład danych. Funkcja `displot` znajduje się na poziomie *figure* (poziom wyższy) i korzysta z funkcji z poziomu *axes* (poziom niższy): `histplot`, `kdeplot`, `ecdfplot` oraz `rugplot`. Do wizualizacji danych można wykorzystywać wszystkie powyższe funkcje z odpowiednimi parametrami.

##### ⭐ Zadanie 5:

Przygotuj histogram (`displot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Zapoznaj się z parametrami `bins` i `binwidth` oraz dobierz dla nich takie wartości, które wg. Ciebie najlepiej wizualizują dane. Zadbaj o czytelność wykresu (tytuł wykresu oraz podpisy osi). 

In [ ]:
winter_height = athlete_events[
    (athlete_events['Season'] == 'Winter') &
    (athlete_events['Sport'] == 'Ski Jumping') &
    (athlete_events['Sex'] == 'M') &
    (athlete_events['Year'] >= 1900)
]
winter_height['Wiek'] = winter_height['Year'].apply(
    lambda x: 'XXI wiek' if x >= 2001 else 'XX wiek'
)

sns.set_theme(style="whitegrid")

g = sns.displot(
    data=winter_height,
    x='Height',
    hue='Wiek',
    kind='hist',
    binwidth=2,
    height=6,
    aspect=1.5
)

g.set_axis_labels("Wzrost (cm)", "Liczba zawodników")
g.fig.suptitle("Rozkład wzrostu mężczyzn w skokach narciarskich", fontsize=14)
plt.tight_layout()
plt.show()

##### ⭐ Zadanie 6:

Przedstaw na wykresie dystrybucji (`displot`) jądrowy estymator gęstości (`kde`) oraz dystrybuantę empiryczną (`ecdf`) dla danych, które wg. Ciebie najlepiej pokażą ich zastosowanie. Każdy rodzaj wykresu dystrybucji musi być przedstawiony na osobnym podwykresie (`subplot`) jednego obrazu. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi i ewentualnie legenda). 

In [ ]:
winter_medalists_age = athlete_events[
    (athlete_events['Season'] == 'Winter') &
    (athlete_events['Medal'].notna()) &
    (athlete_events['Age'].notna()) &
    (athlete_events['Sex'].isin(['M', 'F']))
]

color_map = {"M": "#3498db", "F": "#e74c3c"}

legend_elements = [
    Line2D([0], [0], color=color_map["M"], label='Mężczyźni'),
    Line2D([0], [0], color=color_map["F"], label='Kobiety')
]

sns.set_theme(style="whitegrid")

g = sns.displot(
    data=winter_medalists_age,
    x='Age',
    hue='Sex',
    palette=color_map,
    kind='kde',
    height=5,
    aspect=1.2,
    legend=False,
)

g.fig.suptitle("KDE – Rozkład wieku medalistów zimowych igrzysk wg płci", fontsize=13)
g.set_axis_labels("Wiek", "Gęstość")
plt.legend(title='Płeć', handles=legend_elements, loc='best')
plt.tight_layout()
plt.show()

g2 = sns.displot(
    data=winter_medalists_age,
    x='Age',
    hue='Sex',
    kind='ecdf',
    palette=color_map,
    height=5,
    aspect=1.2,
    legend=False,
)

g2.fig.suptitle("ECDF – Dystrybuanta wieku medalistów zimowych igrzysk wg płci", fontsize=13)
g2.set_axis_labels("Wiek", "Prawdopodobieństwo")
plt.legend(title='Płeć', handles=legend_elements, loc='best')
plt.tight_layout()
plt.show()

##### ⭐ Zadanie 7:

Przygotuj dwuwymiarowy histogram (`displot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi oraz legenda). 

In [ ]:
years_range = list(range(1980, 2021, 2))

top_10_sports_pol = (
    athlete_events[athlete_events['NOC'] == 'POL']
    .groupby('Sport')
    .size()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)
polish_top10_filtered = athlete_events[
    (athlete_events['NOC'] == 'POL') &
    (athlete_events['Sport'].isin(top_10_sports_pol)) &
    (athlete_events['Year'].isin(years_range))
]
sns.set_theme(style="whitegrid")

g = sns.displot(
    data=polish_top10_filtered,
    x='Year',
    y='Sport',
    kind='hist',
    discrete=(True, True),
    cbar=True,
    height=6,
    aspect=1.6
)

sports = sorted(polish_top10_filtered['Sport'].unique())
for i in range(len(sports) - 1):
    g.ax.hlines(
        y=i,
        xmin=1979, xmax=2016,
        colors='gray',
        linestyles='dotted',
        linewidth=0.6,
        alpha=0.7
    )

g.ax.set_xticks(range(1980, 2017, 2))


g.set_axis_labels("Rok igrzysk", "Dyscyplina sportowa")
g.fig.suptitle("Udział Polski w top 10 dyscyplinach na igrzyskach olimpijskich (1980–2016)", fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

--------------------
### Wykresy dystrybucji +

Podstawowe typy wykresów dystrybucji mogą być dodatkowo rozszerzone o więcej informacji. Wizualizacja zależności 2 zmiennych może przedstawiać na marginesach wizualizacje rozkładu każdej zmiennej osobno. Przy wybraniu większej liczby zmiennych, możliwe jest wygenerowanie wszystkich kombinacji pomiędzy nimi. Służą do tego funkcje `jointplot` i `pairplot`, które znajdują się na poziomie *figure*.

##### ⭐ Zadanie 8:

Przygotuj wykres punktowy z rozkładem gęstości na osiach marginalnych (`jointplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Zapoznaj się z obiektem `plot_joint` i narysuj dodatkowo rozkład gęstości wartości na głównym wykresie punktowym. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi oraz legenda). 

In [ ]:
athlete_events['Height'] = pd.to_numeric(athlete_events['Height'], errors='coerce')
athlete_events['Weight'] = pd.to_numeric(athlete_events['Weight'], errors='coerce')

selected_sport = 'Figure Skating'

sport_subset = athlete_events[
    (athlete_events['Sport'] == selected_sport) &
    (athlete_events['Weight'].notna()) &
    (athlete_events['Height'].notna()) &
    (athlete_events['Sex'].notna())
].copy()

color_map = {"M": "#3498db", "F": "#e74c3c"}

sns.set_theme(style="whitegrid")

g = sns.jointplot(
    data=sport_subset,
    x='Weight',
    y='Height',
    kind='scatter',
    palette=color_map,
    hue='Sex',
    marginal_kws=dict(fill=True),
    height=8,
    ratio=4
)

for color in color_map.items():
    g.plot_joint(
        sns.kdeplot,
        levels=4,
        color=color,
        alpha=0.3,
        linewidths=0.8
    )

g.fig.suptitle(f"Relacja wzrostu i wagi łyżwiarzy figurowych", fontsize=14)
g.set_axis_labels("Waga (kg)", "Wzrost (cm)", fontsize=12, labelpad=10)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map["M"],
           markersize=10, label='Mężczyźni'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map["F"],
           markersize=10, label='Kobiety')
]

g.ax_joint.legend(title='Płeć', handles=legend_elements, loc='best')

plt.tight_layout()
plt.show()

##### ⭐ Zadanie 9:

Przygotuj wykres punktowy (`pairplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 serie danych. Zapoznaj się z parametrem `markers` i dobierz dla każdej serii danych inny symbol. Zapoznaj się z parametrem `diag_kind` i ustaw histogram dla wszystkich wykresów na przekątnej. Zadbaj o czytelność wykresu (tytuł wykresu, podpisy osi oraz legenda). 

In [ ]:
my_passion = athlete_events.loc[
    (athlete_events['Sport'].isin(['Volleyball', 'Gymnastics', 'Swimming'])) &
    (athlete_events['Year'] >= 2000) &
    (athlete_events['Season'] >= 'Summer'),
    ['Age', 'Height', 'Weight', 'Sport', 'Sex']
]

my_passion['Height'] = pd.to_numeric(my_passion['Height'], errors='coerce')
my_passion['Weight'] = pd.to_numeric(my_passion['Weight'], errors='coerce')
my_passion = my_passion.dropna()

g = sns.pairplot(
    data=my_passion,
    vars=['Age', 'Height', 'Weight'],
    hue='Sport',
    markers=['o', 's', 'D'],
    height=3,
    aspect=1,
    palette='plasma'
)

g.fig.suptitle('Porównanie parametrów fizycznych zawodników olimpijskich od 2000r', fontsize=14, y=1.02)

labels = ['Wiek', 'Wzrost (cm)', 'Waga (kg)']
for i, ax in enumerate(g.axes.flat):
    ax.tick_params(labelsize=12)
    if i % 3 == 0:
        ax.set_ylabel(labels[i // 3], fontsize=12)
    if i >= 6:
        ax.set_xlabel(labels[i - 6], fontsize=12)

g._legend.set_title('Dyscyplina sportowa')

plt.show()

--------------------
### Wykresy kategorialne

Wykresy kategorialne przedstawiają dane podzielone na kategorie. Funkcja `catplot` znajduje się na poziomie *figure* (poziom wyższy) i korzysta z funkcji z poziomu *axes* (poziom niższy): `stripplot`, `swarmplot`, `boxplot`, `violinplot`, `pointplot` oraz `barplot`. Do wizualizacji danych można wykorzystywać wszystkie powyższe funkcje z odpowiednimi parametrami.

##### ⭐ Zadanie 10: 

Przygotuj wykres punktowy (`catplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 kategorie i co najmniej 2 serie danych. Zapoznaj się z dwiema metodami rysowania wykresów punktowych kategorialnych (`strip`, `swarm`) i zdecyduj, która będzie lepsza w twoim rozwiązaniu. Zapoznaj się z parametrem `order` i dobierz dla niego takie wartości, żeby na osi X etykiety były pokazane w kolejności innej niż domyślna. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
female_alpine_skiing = athlete_events.loc[
    (athlete_events['Sport'].isin(['Basketball', 'Football', 'Volleyball'])) &
    (athlete_events['Sex'] == 'F') &
    (athlete_events['Year'] >= 2000),
    ['Year', 'Sport', 'Height']
]

ball_colors = {
    'Football': '#006400',
    'Basketball': '#E66100',
    'Volleyball': '#005DFF'
}

sns.set(style="whitegrid")
g = sns.catplot(
    x='Year',
    y='Height',
    hue='Sport',
    data=female_alpine_skiing,
    palette=ball_colors,
    kind='swarm',
    height=6,
    aspect=2,
    legend=False
)

g.fig.suptitle("Porównanie wzrostu zawodniczek w trzech popularnych dyscyplinach zespołowych od 2000r.", fontsize=14)
g.set_axis_labels("Rok", "Wzrost (cm)")
g.fig.legend(
    title="Event",
    loc='center',
    handles=[
        Patch(facecolor='#006400', alpha=1, label='Piłka nożna'),
        Patch(facecolor='#E66100', alpha=1, label='Koszykówka'),
        Patch(facecolor='#005DFF', alpha=1, label='Siatkówka'),
    ],
    bbox_to_anchor=(1, 1),
    borderaxespad=0.,
    frameon=False
)

plt.tight_layout()
plt.subplots_adjust(top=0.88)

plt.show()

##### ⭐ Zadanie 11:

Przygotuj wykres pudełkowy (`catplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 kategorie oraz co najmniej 2 serie danych. Zapoznaj się z dwiema metodami rysowania wykresów pudełkowych kategorialnych (`box`, `boxen`) i zdecyduj, która będzie lepsza w twoim rozwiązaniu. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
curling = athlete_events.loc[
          (athlete_events['Sport'] == 'Curling') &
          (athlete_events['Year'] >= 1998)
, :].copy()
curling['Age'] = pd.to_numeric(curling['Age'], errors='coerce')
curling = curling.dropna(subset=['Age', 'Sex'])

medal_order = ['Gold', 'Silver', 'Bronze']
custom_palette = {'Gold': '#FFD700', 'Silver': '#C0C0C0', 'Bronze': '#CD7F32'}

sns.set_theme(style='whitegrid')

g = sns.catplot(
    data=curling,
    x='Sex',
    y='Age',
    hue='Medal',
    hue_order=medal_order,
    palette=custom_palette,
    kind='box',
    height=8,
    aspect=2,
    legend_out=False,
    dodge=True
)

g.set_axis_labels('Płeć', 'Wiek zawodników')
g.fig.suptitle('Rozkład wieku zawodników Curlingu według płci i zdobytych medali od 1998r', fontsize=14, y=1.02)

g.legend.set_title('Medal')
new_labels = {'Gold': 'Złoty', 'Silver': 'Srebrny', 'Bronze': 'Brązowy'}
for t, (k, v) in zip(g.legend.texts, new_labels.items()):
    if k in medal_order:
        t.set_text(v)

g.ax.set_xticks([0, 1])
g.ax.set_xticklabels(['Kobiety', 'Mężczyźni'])
g.ax.set_ylim(15, curling['Age'].max() + 5)

plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

##### ⭐ Zadanie 12:

Przygotuj wykres kolumnowy (`catplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 kategorie oraz co najmniej 2 serie danych. Zapoznaj się z parametrem `palette` i dobierz dla niego nową wartość. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
running = r"^Athletics Men's (100|200|400|800|1,500|5,000|10,000) metres$"
short = r"^Athletics Men's (100|200) metres$"
medium = r"^Athletics Men's (400|800) metres$"
long = r"^Athletics Men's (1,500|5,000|10,000) metres$"

mean_height_medalist = athlete_events.loc[
    (athlete_events['Event'].str.contains(running, regex=True)) &
    (athlete_events['Year'] >= 2000) &
    (athlete_events['Medal'].notna()),
    ['Year', 'Height']
]

mean_height_medalist['distance_category'] = 'Sprint'
mean_height_medalist.loc[
    athlete_events['Event'].str.contains(medium, regex=True),
    ['distance_category']
] = 'Middle distance'
mean_height_medalist.loc[
    athlete_events['Event'].str.contains(long, regex=True),
    ['distance_category']
] = 'Long distance'

mean_height_medalist = mean_height_medalist.groupby(['Year', 'distance_category']).mean().reset_index()

sns.set(style='whitegrid')

colors = {
    'Sprint': '#FF6347',
    'Middle distance': '#FFA500',
    'Long distance': '#4682B4'
}

g = sns.catplot(
    data=mean_height_medalist,
    kind='bar',
    x='Year',
    y='Height',
    hue='distance_category',
    palette=colors,
    height=6,
    aspect=2,
    legend=False,
)

g.fig.suptitle("Średni wzrost medalistów w biegach (mężczyźni) od 2000 roku", fontsize=14)
g.set_axis_labels("Rok", "Wzrost (cm)")
g.fig.legend(
    title="Rodzaj dystansu",
    loc='center',
    handles=[
        Patch(facecolor='#FF6347', alpha=1, label='Sprint'),
        Patch(facecolor='#FFA500', alpha=1, label='Średni dystans'),
        Patch(facecolor='#4682B4', alpha=1, label='Długi dystans'),
    ],
    bbox_to_anchor=(1, 1),
    borderaxespad=0.,
    frameon=False
)
plt.yticks(range(0, 196, 15))

plt.tight_layout()
plt.subplots_adjust(top=0.88)

plt.show()

##### ⭐ Zadanie 13:

Przygotuj wykres kolumnowy (`catplot`) opierając się na danych, które sam wybierzesz w sensowny sposób. Uwzględnij co najmniej 3 kategorie oraz co najmniej 2 serie danych. Porównaj ze sobą cztery rodzaje słupków błędu: błąd standardowy (`se` - standard error), odchylenie standardowe (`sd` - standard deviation), przedział centylowy (`pi` - percentile interval) i przedział ufności (`ci` - confidence interval). Każdy rodzaj słupków błędu musi być przedstawiony na osobnym podwykresie (`subplot`) jednego obrazu. Zadbaj o czytelność wykresu (tytuł wykresu i podpisy osi). 

In [ ]:
errorbars = ['ci', 'pi', 'se', 'sd']
error_names = {
    'ci': 'Przedział ufności (CI)',
    'pi': 'Przedział centylowy (PI)',
    'se': 'Błąd standardowy (SE)',
    'sd': 'Odchylenie standardowe (SD)'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharey=True)

for i, error in enumerate(errorbars):
    x, y = divmod(i, 2)
    sns.barplot(
        data=mean_height_medalist,
        x='Year',
        y='Height',
        errorbar=error,
        ax=axes[x, y],
        legend=False,
        alpha=0.5,
    )
    axes[x, y].set_title(error_names[error], fontsize=12)
    axes[x, y].set_xlabel("Rok" if i >= 2 else "")
    axes[x, y].set_ylabel("Wzrost (cm)" if i % 2 == 0 else "")
    axes[x, y].set_yticks(range(150, 191, 5))
    axes[x, y].set_ylim(bottom=150, top=190)

fig.suptitle("Porównanie metod wizualizacji niepewności dla średniego wzrostu medalistów (mężczyźni, biegi)", fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()